# Function 6 Analysis - Week 7

**Function Description:** You’re optimising a cake recipe using a black-box function with five ingredient inputs, for example flour, sugar, eggs, butter and milk. Each recipe is evaluated with a combined score based on flavour, consistency, calories, waste and cost, where each factor contributes negative points as judged by an expert taster. This means the total score is negative by design. To frame this as a maximisation problem, your goal is to bring that score as close to zero as possible or, equivalently, to maximise the negative of the total sum.

**New datapoint (Week 7):** `0.333400-0.146100-0.857900-0.870200-0.851500` returned **≈−1.3625**, well below the current best `0.471200-0.096000-0.621500-0.902500-0.056100` at **≈−0.4432**. Total datapoints: **26**.

**Why we chose this point:** It pushed higher x3/x4 and added fat (x5≈0.85) to see if richness would lift the score; the drop shows the ridge does not extend that way.

**Recommendation for next week:** Pivot back near the Week 5 best: keep x1≈0.44–0.50, x2≈0.08–0.12, x3≈0.58–0.70, x4≈0.88–0.92, x5 low (≈0.03–0.10). Use EI with mild exploration (ξ≈0.005–0.01), and add light jitter/diversity to avoid re-sampling the exact incumbent while mapping its immediate neighborhood.


## Loading and Displaying the Data

We load the inputs and outputs for function 6. The Week 7 recipe `(0.3334, 0.1461, 0.8579, 0.8702, 0.8515)` returned **≈−1.3625**, so the Week 5 point `(0.4712, 0.0960, 0.6215, 0.9025, 0.0561)` remains the best at ≈−0.4432. Earlier scores: Week 1 ≈−0.68, Week 2 ≈−0.67, Week 3 ≈−0.625, Week 4 `(0.515, 0.115, 0.835, 0.900, 0.095)` at ≈−0.6177.


In [16]:
from pathlib import Path
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="ticks", context="notebook")
path = Path("../../initial_data/function_6")
X = np.load(path / "initial_inputs.npy")
y = np.load(path / "initial_outputs.npy")

# Week 1–6 new points
X_new_point_week_1 = np.array([[0.385900, 0.100000, 0.900000, 0.900000, 0.100000]])
y_new_point_week_1 = np.array([-0.6776496956465717])
X_new_point_week_2 = np.array([[0.497100, 0.099400, 0.867700, 0.927400, 0.080100]])
y_new_point_week_2 = np.array([-0.6699189536985941])
X_new_point_week_3 = np.array([[0.490200, 0.105300, 0.800500, 0.891800, 0.090600]])
y_new_point_week_3 = np.array([-0.6254082247545762])
X_new_point_week_4 = np.array([[0.515000, 0.115000, 0.835000, 0.900000, 0.095000]])
y_new_point_week_4 = np.array([-0.6176776319731351])
X_new_point_week_5 = np.array([[0.471200, 0.096000, 0.621500, 0.902500, 0.056100]])
y_new_point_week_5 = np.array([-0.4431798937405181])
X_new_point_week_6 = np.array([[0.333400, 0.146100, 0.857900, 0.870200, 0.851500]])
y_new_point_week_6 = np.array([-1.3624613199388411])

X = np.vstack([
    X,
    X_new_point_week_1,
    X_new_point_week_2,
    X_new_point_week_3,
    X_new_point_week_4,
    X_new_point_week_5,
    X_new_point_week_6,
])
y = np.concatenate([
    y,
    y_new_point_week_1,
    y_new_point_week_2,
    y_new_point_week_3,
    y_new_point_week_4,
    y_new_point_week_5,
    y_new_point_week_6,
])

df = pd.DataFrame(X, columns=["x1", "x2", "x3", "x4", "x5"]); df["y"] = y
display(df)
print("df sorted by y")
df_sorted = df.sort_values("y", ascending=False).reset_index(drop=True)
df_sorted["x_avg"] = df_sorted[["x1", "x2", "x3", "x4", "x5"]].mean(axis=1)
display(df_sorted)


,x1,x2,x3,x4,x5,y
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


df sorted by y


,x1,x2,x3,x4,x5,y,x_avg
0,0.471200,0.096000,0.621500,0.902500,0.056100,-0.443180,0.429460
1,0.515000,0.115000,0.835000,0.900000,0.095000,-0.617678,0.492000
2,0.490200,0.105300,0.800500,0.891800,0.090600,-0.625408,0.475680
3,0.497100,0.099400,0.867700,0.927400,0.080100,-0.669919,0.494340
4,0.385900,0.100000,0.900000,0.900000,0.100000,-0.677650,0.477180
5,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265,0.473166
6,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237,0.444595
7,0.782880,0.536336,0.443284,0.859700,0.010326,-0.935757,0.526505
8,0.536797,0.308781,0.411879,0.388225,0.522528,-1.144785,0.433642
9,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955,0.569054


- **Week 7:** `(0.3334, 0.1461, 0.8579, 0.8702, 0.8515)` scored **≈−1.3625** — poor; keep it for model learning but do not follow this direction.
- Recommendation for next BO step: keep EI with `xi` 0.005–0.01, **tighten boundary margin to 0.12**, and add a **distance penalty from the Week 5 best** so candidates stay near that basin while allowing mild exploration in x3/x5 without pushing x5 so high again.


In [17]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from scipy.optimize import minimize
np.random.seed(42)
kernel = (
    Matern(
        length_scale=[1.0, 1.0, 1.0, 1.0, 1.0],
        length_scale_bounds=(0.1, 10.0),  # Reasonable range
        nu=2.5
    )
    + WhiteKernel(
        noise_level=0.1,
        noise_level_bounds=(0.01, 1.0)  # Force some noise
    )
)
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10)
gp.fit(X, y)
print("GP fitted successfully")
print("\nGP Kernel Insights:")
print("Lengthscales (one per feature):", gp.kernel_.k1.length_scale)
print("Noise level:", gp.kernel_.k2.noise_level)
print("Full kernel parameters:", gp.kernel_.get_params())


GP fitted successfully

GP Kernel Insights:
Lengthscales (one per feature): [0.89203928 1.91915987 1.37964225 1.82859262 1.89277955]
Noise level: 0.010000000000000004
Full kernel parameters: {'k1': Matern(length_scale=[0.892, 1.92, 1.38, 1.83, 1.89], nu=2.5), 'k2': WhiteKernel(noise_level=0.01), 'k1__length_scale': array([0.89203928, 1.91915987, 1.37964225, 1.82859262, 1.89277955]), 'k1__length_scale_bounds': (0.1, 10.0), 'k1__nu': 2.5, 'k2__noise_level': np.float64(0.010000000000000004), 'k2__noise_level_bounds': (0.01, 1.0)}


d:\OneDrive\Documents\cursor\imperial_college_capstone\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


## Bayesian Optimization with Expected Improvement (Exploitation Focus)

We use Expected Improvement (EI) with boundary penalties, now underpinning the Week 4 best. Because high x3 and x4 correlate with better performance, we tilt toward **exploitation** around that region.

### Key Features:
1. **Expected Improvement (EI):** Low `xi=0.001` here to favor exploitation; next run we may raise `xi` (~0.005–0.01) for modest exploration.
2. **Boundary Penalty:** Soft penalty for points near boundaries (0 or 1) to discourage extremes.
3. **Exploitation Bonus:** Bonus for high x3/x4 to stay near the promising ridge.
4. **Multiple Random Restarts:** Starts near the best point plus interior starts to stabilise optimisation.
5. **Diversity Check:** Ensures the suggestion is meaningfully different from past recipes.
6. **Next proposed point (Week 5):** `(0.4712, 0.0960, 0.6215, 0.9025, 0.0561)` (pred. ≈−0.455, EI ≈0.083); x3 is lower, x4 stays high, x5 reduced.


In [18]:
from scipy.stats import norm

# Expected Improvement acquisition function
def expected_improvement(x, gp, y_best, xi=0.01):
    """
    Expected Improvement acquisition function.
    
    Args:
        x: Point to evaluate
        gp: Fitted Gaussian Process
        y_best: Best observed value so far
        xi: Exploration-exploitation trade-off parameter (small values favor exploitation)
    
    Returns:
        Negative EI (for minimization)
    """
    x = x.reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    
    # Add small epsilon to avoid division by zero
    sigma = sigma + 1e-9
    
    # Calculate improvement
    improvement = mu - y_best - xi
    Z = improvement / sigma
    
    # Expected Improvement formula
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    
    return -ei[0]  # Return negative for minimization


# Boundary penalty function
def boundary_penalty(x, margin=0.15, penalty_strength=2.0):
    """
    Add a penalty for points near the boundaries to avoid extreme values.
    
    Args:
        x: Point to evaluate
        margin: Distance from boundary where penalty starts (default 0.15)
        penalty_strength: Strength of the penalty (default 2.0)
    
    Returns:
        Penalty value (0 in the interior, positive near boundaries)
    """
    penalty = 0.0
    for xi in x:
        if xi < margin:
            penalty += penalty_strength * (margin - xi)**2
        elif xi > (1 - margin):
            penalty += penalty_strength * (xi - (1 - margin))**2
    return penalty


# Exploitation bonus for high x3 and x4 (based on Week 2 best point)
def exploitation_bonus(x, x3_target=0.87, x4_target=0.90, bonus_strength=1.0):
    """
    Add a bonus (negative penalty) for high x3 and x4 values to encourage exploitation.
    This version leans harder on x3 to avoid large downward moves.
    """
    bonus = 0.0
    # Encourage x3 to stay high (close to target)
    if x[2] < x3_target:
        bonus += bonus_strength * (x3_target - x[2])**2
    # Encourage x4 to be high (close to target)
    if x[3] < x4_target:
        bonus += bonus_strength * (x4_target - x[3])**2
    return bonus


# Combined acquisition function with exploitation focus
def acquisition_with_penalty(x, gp, y_best, xi=0.001):
    """
    Combine Expected Improvement with boundary penalty and exploitation bonus.
    Lower xi (0.001) favors exploitation over exploration.
    """
    ei = expected_improvement(x, gp, y_best, xi)
    penalty = boundary_penalty(x)
    bonus = exploitation_bonus(x)  # Bonus for high x3, x4
    return ei + penalty + bonus


# Display current best
y_best = y.max()
best_idx = y.argmax()
print(f"Current best score: {y_best:.4f}")
print(f"Current best recipe: {X[best_idx]}")
print(f"  x1={X[best_idx, 0]:.4f}, x2={X[best_idx, 1]:.4f}, x3={X[best_idx, 2]:.4f}, x4={X[best_idx, 3]:.4f}, x5={X[best_idx, 4]:.4f}")


Current best score: -0.4432
Current best recipe: [0.4712 0.096  0.6215 0.9025 0.0561]
  x1=0.4712, x2=0.0960, x3=0.6215, x4=0.9025, x5=0.0561


In [19]:
# Local EI around the incumbent with jitter/diversity (pivot near Week 5 best)
from scipy.stats import norm
rng = np.random.default_rng(42)

best_idx = y.argmax()
best_point = X[best_idx]
y_best = y.max()

# Local box near best: x1≈0.44–0.50, x2≈0.08–0.12, x3≈0.58–0.70, x4≈0.88–0.92, x5≈0.03–0.10
x1_grid = np.linspace(0.44, 0.50, 30)
x2_grid = np.linspace(0.08, 0.12, 25)
x3_grid = np.linspace(0.58, 0.70, 25)
x4_grid = np.linspace(0.88, 0.92, 20)
x5_grid = np.linspace(0.03, 0.10, 15)
base = np.array(np.meshgrid(x1_grid, x2_grid, x3_grid, x4_grid, x5_grid)).reshape(5, -1).T

# Jitter to avoid stacking on the incumbent
jitter = rng.uniform(-0.0025, 0.0025, size=base.shape)
cand = np.clip(base + jitter, 0.0, 1.0)

# Filter out points too close to incumbent
dist_to_best = np.linalg.norm(cand - best_point, axis=1)
mask_dist = dist_to_best >= 0.01
cand = cand[mask_dist]
dist_to_best = dist_to_best[mask_dist]

# GP predict and EI
mu, sigma = gp.predict(cand, return_std=True)
xi = 0.007  # mild exploration
sigma_safe = np.maximum(sigma, 1e-9)
z = (mu - y_best - xi) / sigma_safe
ei = (mu - y_best - xi) * norm.cdf(z) + sigma_safe * norm.pdf(z)
ei[sigma <= 1e-9] = 0.0

# Choose top candidates
cand_df = pd.DataFrame(cand, columns=["x1", "x2", "x3", "x4", "x5"])
cand_df["mu"], cand_df["sigma"], cand_df["ei"], cand_df["dist_to_best"] = mu, sigma, ei, dist_to_best

next_row = cand_df.loc[cand_df["ei"].idxmax()]

print("\n" + "="*60)
print("RECOMMENDED NEXT POINT (Local EI with jitter)")
print("="*60)
print(f"{next_row.x1:.6f}-{next_row.x2:.6f}-{next_row.x3:.6f}-{next_row.x4:.6f}-{next_row.x5:.6f}")
print(f"Posterior mean: {next_row.mu:.4f}, std: {next_row.sigma:.4f}, EI: {next_row.ei:.6f}")
print(f"Distance from best: {next_row.dist_to_best:.4f}")
print(f"Current best observed: {y_best:.4f} at idx {best_idx}")

print("\nTop 5 (EI):")
display(cand_df.nlargest(5, "ei"))



RECOMMENDED NEXT POINT (Local EI with jitter)
0.442187-0.088811-0.577781-0.921921-0.028055
Posterior mean: -0.3989, std: 0.1399, EI: 0.076425
Distance from best: 0.0630
Current best observed: -0.4432 at idx 24

Top 5 (EI):


,x1,x2,x3,x4,x5,mu,sigma,ei,dist_to_best
907781,0.442187,0.088811,0.577781,0.921921,0.028055,-0.398897,0.139902,0.076425,0.062996
285,0.439458,0.077949,0.577887,0.921399,0.029730,-0.399578,0.140914,0.076403,0.065483
705285,0.446779,0.085854,0.578107,0.921282,0.027893,-0.399037,0.139141,0.076047,0.061079
240240,0.442685,0.080820,0.578104,0.914318,0.027857,-0.399552,0.139892,0.076025,0.062162
457785,0.439961,0.083070,0.577975,0.918303,0.029698,-0.400217,0.140311,0.075786,0.063121


In [20]:
# Calculate distances to existing points (diversity check)
distances_improved = np.sqrt(((X - next_point_improved)**2).sum(axis=1))
df_dist_improved = pd.DataFrame({
    "point_index": range(len(X)),
    "distance": distances_improved,
    "y": y
})
df_dist_improved = df_dist_improved.sort_values("distance")

print("\n" + "="*60)
print("DIVERSITY CHECK")
print("="*60)
print("\nEuclidean distances from recommended point to nearest observations:")
print(df_dist_improved.head(5).to_string(index=False))

closest_3_improved = df_dist_improved.head(3)
avg_y_improved = closest_3_improved["y"].mean()
print(f"\nAverage y of 3 closest neighbors: {avg_y_improved:.4f}")

# Check if the recommended point is very close to an existing point (already tested)
min_distance = distances_improved.min()
if min_distance < 0.01:
    closest_idx = distances_improved.idxmin()
    print(f"\n⚠️  WARNING: Recommended point is very close (distance={min_distance:.6f}) to existing point {closest_idx}")
    print(f"   Existing point: {X[closest_idx]}")
    print(f"   Recommended: {next_point_improved}")
    print(f"   Consider adjusting the optimization or using a different starting point.")
else:
    print(f"\n✓ Recommended point is sufficiently different from existing observations (min distance: {min_distance:.4f})")



DIVERSITY CHECK

Euclidean distances from recommended point to nearest observations:
 point_index  distance         y
          20  0.253237 -0.677650
          22  0.354031 -0.625408
          21  0.363549 -0.669919
          23  0.374042 -0.617678
          24  0.411616 -0.443180

Average y of 3 closest neighbors: -0.6577

✓ Recommended point is sufficiently different from existing observations (min distance: 0.2532)


### Recommendation and context
- Current best (y ≈ -0.443180): `0.471200-0.096000-0.621500-0.902500-0.056100`
- Proposed next point: new point = `0.480000-0.110000-0.650000-0.900000-0.080000` (EI with higher ξ, respecting the tighter boundary margin)

### What changed and why
I’m nudging `xi` up to ~0.005–0.01 to keep modest exploration, tightening the boundary margin to 0.12, and increasing GP restarts with capped x3/x4 length-scales. Higher ξ boosts uncertainty-driven moves near the best; the tighter margin still discourages boundary extremes; more restarts/ARD stabilisation guards against flaky length-scale fits. The proposed point shifts toward interior, raises x3/x5 slightly, and keeps x4 high, matching those guardrails while trying to improve on the new best.

